# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"License: {meta.license}")
print(f"Published: {meta.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use the dataset schema to discover the available record sets and their fields (columns). All Croissant objects are referenced by their `@id`.

In [ ]:
# List all available record sets and fields by their @id
record_sets = list(dataset.metadata.recordSets)
print("Available Record Sets (by @id and name):\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, name: {rs.get('name')}")
    fields = rs.get('fields', [])
    if fields:
        print("  Fields / columns (by @id and name):")
        for field in fields:
            # field is a dict with @id
            print(f"    - {field['@id']}: {field.get('name','')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. Here, we use the main protein record set as an example.

In [ ]:
# Extract all dataframes for each discovered record set
dataframes = {}
for rs in dataset.metadata.recordSets:
    rs_id = rs['@id']
    print(f"Loading records for {rs_id}...", end=' ')
    # Try loading; handle if records are empty
    try:
        recs = list(dataset.records(record_set=rs_id))
        if recs:
            df = pd.DataFrame(recs)
            dataframes[rs_id] = df
            print(f"✔️ Loaded {len(df)} rows with columns: {df.columns.tolist()}")
        else:
            print('⚠️ No records found.')
    except Exception as e:
        print(f"⚠️ Error: {e}")

# For demonstration, pick the main record set (by domain knowledge, often the first, or has 'Protein' in name or id)
if dataframes:
    primary_rs_id = list(dataframes.keys())[0]
    print(f"\nPrimary record set for analysis: {primary_rs_id}")
    print(dataframes[primary_rs_id].head())
else:
    primary_rs_id = None
    print("No dataframes extracted.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes outlier removal, transformations, and basic grouping operations.

> **Note:** All field and record set references use their Croissant `@id` as required.

In [ ]:
# Example EDA: Filter and Normalize
if primary_rs_id is not None:
    df = dataframes[primary_rs_id].copy()
    print(f"Columns in {primary_rs_id}: {df.columns.tolist()}")

    # Try to identify a usable numeric field by name or by checking dtype
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    # Fallback: check for common names
    if not numeric_candidates:
        for candidate in ['coverage_percent', 'MW_kDa', 'Abundance', 'peptide_count', 'unique_peptide_count', 'pI']:
            if candidate in df.columns:
                numeric_candidates.append(candidate)

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field: {numeric_field} (for filtering and normalization)")
        # Set a filter threshold (e.g., top 20th percentile)
        threshold = df[numeric_field].quantile(0.8) if df[numeric_field].dtype.kind in 'fi' else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
        display_cols = [numeric_field]
        print(filtered_df[display_cols].head())

        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Try grouping by a categorical field (e.g., 'sample', 'group', 'protein_type'). Fallback to first object column.
        cat_field = None
        for c in df.columns:
            if df[c].dtype==object and c != numeric_field:
                cat_field = c
                break
        if cat_field:
            grouped_df = filtered_df.groupby(cat_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(f"\nGrouped mean {numeric_field} by {cat_field} (top 5):")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No primary record set selected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and pandas plotting.

In [ ]:
# Visualization example: Histogram and boxplot for the numeric field
if primary_rs_id is not None and numeric_field in dataframes[primary_rs_id].columns:
    plt.figure(figsize=(14,5))
    plt.subplot(1,2,1)
    dataframes[primary_rs_id][numeric_field].hist(bins=20)
    plt.title(f"{numeric_field} Distribution")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.subplot(1,2,2)
    dataframes[primary_rs_id][numeric_field].plot.box()
    plt.title(f"{numeric_field} Boxplot")
    plt.show()

    # If grouped_df exists
    if 'grouped_df' in locals():
        grouped_df.plot(kind='bar', x=cat_field, y=numeric_field, legend=False)
        plt.title(f"Mean {numeric_field} by {cat_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(cat_field)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook we demonstrated how to use the `mlcroissant` library to load and explore the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells.

We programmatically inspected the available record sets and fields using their Croissant `@id`, loaded data into pandas DataFrames, performed EDA including filtering, normalization, and grouping, and visualized the data. The Croissant schema's rich metadata, paired with `mlcroissant`, enables transparent, referenceable and reproducible data science workflows.

**Next steps:** This exploration can be extended with domain-specific analyses, advanced statistics, or machine learning, using the well-documented field `@id` references.